# CARec encoder and dimension study

- Load the saved Amazon Electronics catalogue, interaction partitions, and ID maps.
- Use the `NI-PML benchmarks` kernel from the project `.venv`.
- Data exploration, text preparation, sampling, splitting, and saving belong in [data_exploration.ipynb](data_exploration.ipynb).


In [24]:
from pathlib import Path
import gzip
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


## Load products and ID maps

- Read one product record per JSONL line.
- Load user and product ID maps, keeping their IDs as strings.


In [25]:
items_path = PREPARED_DIR / "items.jsonl.gz"
product_records = []

with gzip.open(items_path, "rt", encoding="utf-8") as source:
    for line in source:
        product = json.loads(line)
        product_records.append(product)

items = pd.DataFrame(product_records)
del product_records

user_map_path = PREPARED_DIR / "user_ids.tsv.gz"
user_map = pd.read_csv(user_map_path, sep="\t", dtype={"user_id": str})

item_map_path = PREPARED_DIR / "item_ids.tsv.gz"
item_map = pd.read_csv(item_map_path, sep="\t", dtype={"item_id": str})


## Load interaction partitions

- Load train, validation, test, and cold interactions.
- Keep user and product IDs as strings and preserve float precision.


In [26]:
partition_names = ("train", "valid", "test", "cold")

interaction_types = {
    "user_id:token": str,
    "item_id:token": str,
}
partitions = {}

for name in partition_names:
    partition_path = PREPARED_DIR / f"{name}.tsv.gz"
    frame = pd.read_csv(
        partition_path,
        sep="\t",
        dtype=interaction_types,
        float_precision="round_trip",
    )
    partitions[name] = frame

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


## Show the loaded data

- Print product, user, and partition counts.
- Show two product texts.


In [27]:
print(f"Loaded {len(items):,} products and {len(user_map):,} users")

for name, frame in partitions.items():
    print(f"{name}: {len(frame):,} interactions")

text_preview = items[["item_id:token", "text"]].head(2)
display(text_preview)


Loaded 34,220 products and 81,520 users
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...


## Encoder candidates

- `text` is the saved product text from `items["text"]`; `instruction` describes the recommendation task. The pilot instruction below is provisional.
- Run the three instruction models with and without instructions: plain `text` for Qwen/E5-instruct, an empty prompt for INSTRUCTOR. Pilot these variants before bulk generation.
- E5-base's `query: ` prefix is required formatting, not a task instruction. `\n` below means a newline.

| Model | Instruction-based | Input format |
| --- | --- | --- |
| [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | No | `text` |
| [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | No | `text` |
| [e5-base-v2](https://huggingface.co/intfloat/e5-base-v2) | No | `query: {text}` |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | Yes | `Instruct: {instruction}\nQuery:{text}` |
| [multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct) | Yes | `Instruct: {instruction}\nQuery: {text}` |
| [instructor-base](https://huggingface.co/hkunlp/instructor-base) | Yes | `text` with `prompt=instruction`, excluding the prompt from pooling |


## One-batch encoder pilots

- Use the same four saved products for every encoder. Keep native dimensions and normalize outputs to float32.
- Each section shows parameters and one exact input before loading the model. Run its pilot cell separately.
- Test both instruction modes for Qwen, E5-instruct, and INSTRUCTOR. The instruction is a pilot choice.
- Download weights on first use into the ignored project cache. Release each model after its section.
- Use MPS when available. Set `PILOT_DEVICE = "cpu"` to test CPU explicitly; failures stay visible.
- These small batches check inference compatibility, not catalogue throughput or recommendation quality. No full-catalogue embeddings are saved.
- INSTRUCTOR uses Sentence Transformers' [prompt handling](https://www.sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html), with instruction tokens excluded from pooling.


### MiniLM-L6-v2

- 22.7 million parameters, 6 Transformer layers.
- Output dimension: 384.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 256 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).


In [28]:
import torch
from sentence_transformers import SentenceTransformer

model_id = "sentence-transformers/all-MiniLM-L6-v2"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 4

sample_texts = items["text"].head(batch_size).tolist()
print(f"Device: {device}")
print(sample_texts[0])


Device: mps
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [29]:
minilm = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
minilm_embeddings = minilm.encode(
    sample_texts,
    batch_size=batch_size,
    prompt="",
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(minilm_embeddings.shape)


(4, 384)


### MPNet-base-v2

- 109.5 million parameters, 12 Transformer layers.
- Output dimension: 768.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 384 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-mpnet-base-v2).


In [30]:
import torch
from sentence_transformers import SentenceTransformer

model_id = "sentence-transformers/all-mpnet-base-v2"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 4

sample_texts = items["text"].head(batch_size).tolist()
print(f"Device: {device}")
print(sample_texts[0])


Device: mps
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [31]:
mpnet = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
mpnet_embeddings = mpnet.encode(
    sample_texts,
    batch_size=batch_size,
    prompt="",
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(mpnet_embeddings.shape)


(4, 768)


### E5-base-v2

- [Model card](https://huggingface.co/intfloat/e5-base-v2).


In [10]:
e5base_config = {
    "model_id": "intfloat/e5-base-v2",
    "dimension": 768,
    "instruction_based": False,
}
e5base_prompts = {"query_prefix": "query: "}
preview_encoder(e5base_config, e5base_prompts)



query_prefix, item 0545105668:
query: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


model_id             intfloat/e5-base-v2
dimension                            768
instruction_based                  False
Name: parameters, dtype: object

In [11]:
run_encoder_pilot(e5base_config, e5base_prompts)


Loaded in 2.2s; token limit: 512
query_prefix: OK, (4, 768), float32, 0.06s


### Qwen3-Embedding-0.6B

- [Model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).
- Use eager attention on MPS. The default SDPA path returned non-finite vectors in this environment.


In [12]:
qwen_config = {
    "model_id": "Qwen/Qwen3-Embedding-0.6B",
    "dimension": 1024,
    "instruction_based": True,
    "attention": "eager",
}
qwen_prompts = {
    "without_instruction": "",
    "with_instruction": f"Instruct: {PILOT_INSTRUCTION}\nQuery:",
}
preview_encoder(qwen_config, qwen_prompts)



without_instruction, item 0545105668:
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

with_instruction, item 0545105668:
Instruct: Represent this product for personalized recommendation.
Query:Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


model_id             Qwen/Qwen3-Embedding-0.6B
dimension                                 1024
instruction_based                         True
attention                                eager
Name: parameters, dtype: object

In [13]:
run_encoder_pilot(qwen_config, qwen_prompts)


Loaded in 4.7s; token limit: 32768
without_instruction: OK, (4, 1024), float32, 0.37s
with_instruction: OK, (4, 1024), float32, 0.31s


### Multilingual-E5-large-instruct

- [Model card](https://huggingface.co/intfloat/multilingual-e5-large-instruct).


In [14]:
e5instruct_config = {
    "model_id": "intfloat/multilingual-e5-large-instruct",
    "dimension": 1024,
    "instruction_based": True,
}
e5instruct_prompts = {
    "without_instruction": "",
    "with_instruction": f"Instruct: {PILOT_INSTRUCTION}\nQuery: ",
}
preview_encoder(e5instruct_config, e5instruct_prompts)



without_instruction, item 0545105668:
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

with_instruction, item 0545105668:
Instruct: Represent this product for personalized recommendation.
Query: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


model_id             intfloat/multilingual-e5-large-instruct
dimension                                               1024
instruction_based                                       True
Name: parameters, dtype: object

In [15]:
run_encoder_pilot(e5instruct_config, e5instruct_prompts)


Loaded in 4.0s; token limit: 512
without_instruction: OK, (4, 1024), float32, 0.10s
with_instruction: OK, (4, 1024), float32, 0.11s


### INSTRUCTOR-base

- [Model card](https://huggingface.co/hkunlp/instructor-base).


In [16]:
instructor_config = {
    "model_id": "hkunlp/instructor-base",
    "dimension": 768,
    "instruction_based": True,
    "exclude_prompt": True,
}
instructor_prompts = {
    "without_instruction": "",
    "with_instruction": PILOT_INSTRUCTION + " ",
}
preview_encoder(instructor_config, instructor_prompts)



without_instruction, item 0545105668:
Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.

with_instruction, item 0545105668:
Represent this product for personalized recommendation. Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


model_id             hkunlp/instructor-base
dimension                               768
instruction_based                      True
exclude_prompt                         True
Name: parameters, dtype: object

In [17]:
run_encoder_pilot(instructor_config, instructor_prompts)


Loaded in 2.6s; token limit: 512
without_instruction: OK, (4, 768), float32, 0.11s
with_instruction: OK, (4, 768), float32, 0.07s


## Pilot results

- Show successful conditions from this kernel session. Failed cells keep their traceback; missing conditions have not passed.
- Timing includes the first inference call and is not a bulk-generation estimate.


In [18]:
pilot_summary = pd.DataFrame(pilot_results.values())
display(pilot_summary)
print(f"Successful conditions: {len(pilot_results)}/9")


Successful conditions: 9/9


,model,mode,device,shape,dtype,max_tokens,attention,revision,load_seconds,batch_seconds
0,sentence-transformers/all-MiniLM-L6-v2,plain,mps,"(4, 384)",float32,256,sdpa,1110a243fdf4706b3f48f1d95db1a4f5529b4d41,1.966697,0.123557
1,sentence-transformers/all-mpnet-base-v2,plain,mps,"(4, 768)",float32,384,eager,e8c3b32edf5434bc2275fc9bab85f82640a19130,2.181368,0.074958
2,intfloat/e5-base-v2,query_prefix,mps,"(4, 768)",float32,512,sdpa,f52bf8ec8c7124536f0efb74aca902b2995e5bcd,2.182988,0.055437
3,Qwen/Qwen3-Embedding-0.6B,without_instruction,mps,"(4, 1024)",float32,32768,eager,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,4.684394,0.365076
4,Qwen/Qwen3-Embedding-0.6B,with_instruction,mps,"(4, 1024)",float32,32768,eager,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,4.684394,0.310604
5,intfloat/multilingual-e5-large-instruct,without_instruction,mps,"(4, 1024)",float32,512,sdpa,274baa43b0e13e37fafa6428dbc7938e62e5c439,3.987115,0.104713
6,intfloat/multilingual-e5-large-instruct,with_instruction,mps,"(4, 1024)",float32,512,sdpa,274baa43b0e13e37fafa6428dbc7938e62e5c439,3.987115,0.106843
7,hkunlp/instructor-base,without_instruction,mps,"(4, 768)",float32,512,eager,0b2f22542d2bd6d60dbff22d22f12c15e6562736,2.649472,0.112632
8,hkunlp/instructor-base,with_instruction,mps,"(4, 768)",float32,512,eager,0b2f22542d2bd6d60dbff22d22f12c15e6562736,2.649472,0.070670
